In [18]:
from pathlib import Path
timeseries_data = "sr"
base_dir_ = Path("/home/jovyan/work-easi-eds/data/compat/files")

base_dir = base_dir_ / timeseries_data
#base_dir = Path("/home/jovyan/data/compat/files")
assert base_dir.exists(), f"Missing base directory: {base_dir}"

tile_dirs = sorted([p for p in base_dir.iterdir() if p.is_dir()])

print(f"Found {len(tile_dirs)} processed tiles:\n")

for t in tile_dirs:
    n_files = sum(1 for _ in t.rglob("*") if _.is_file())
    print(f"{t.name:<12}  files: {n_files}")

print("\nTile directories:")
for t in tile_dirs:
    print(" ", t)


Found 1 processed tiles:

p089r078      files: 0

Tile directories:
  /home/jovyan/work-easi-eds/data/compat/files/sr/p089r078


In [19]:
from pathlib import Path

tile = "p089r078"
tile_dir = base_dir / tile
print("Tile dir:", tile_dir)
print("Exists:", tile_dir.exists())

files = sorted([p for p in tile_dir.rglob("*") if p.is_file()])
print("File count:", len(files))

for p in files[:50]:
    print(p.relative_to(tile_dir))


Tile dir: /home/jovyan/work-easi-eds/data/compat/files/sr/p089r078
Exists: True
File count: 0


In [15]:
# from pathlib import Path

# tile = "p089r084"  # <-- change if needed
# tile_dir = Path("/home/jovyan/data/compat/files") / tile
# assert tile_dir.exists(), f"Missing: {tile_dir}"
# print("Tile dir:", tile_dir)

In [16]:
import re
from collections import defaultdict


def human_mb(n_bytes: int) -> float:
    return n_bytes / (1024**2)


# regex helpers
re_pair = re.compile(r"_d(\d{8})(\d{8})_")  # e.g. ..._d2023072820240831_...
re_single = re.compile(r"_(\d{8})_")  # e.g. ..._20230728_db8mz...


def img_type(name: str) -> str:
    """
    Returns the product/type token from the end of the filename, e.g.
    db8mz, dc4mz, dllmz, dljmz (and keeps things like vi-fpc_dljmz)
    """
    stem = Path(name).stem  # removes .img
    # take the last underscore chunk as base type
    last = stem.split("_")[-1]
    # special: keep vi-fpc_dljmz or vi-fpc_dllmz together if present
    if "vi-fpc" in stem:
        if stem.endswith("dljmz"):
            return "vi-fpc_dljmz"
        if stem.endswith("dllmz"):
            return "vi-fpc_dllmz"
    return last


def extract_sort_key(p: Path):
    """
    Returns (kind, date1, date2) for sorting:
      - kind: "pair" or "single" or "none"
      - date1/date2: ints YYYYMMDD (or None)
    """
    name = p.name
    m = re_pair.search(name)
    if m:
        d1, d2 = int(m.group(1)), int(m.group(2))
        return ("pair", d1, d2)
    m = re_single.search(name)
    if m:
        d = int(m.group(1))
        return ("single", d, None)
    return ("none", None, None)


# collect
imgs = sorted(tile_dir.rglob("*.img"))

groups = defaultdict(list)
for p in imgs:
    groups[img_type(p.name)].append(p)

# print grouped + sorted
print(f"Found {len(imgs)} .img files\n")

for t in sorted(groups.keys()):
    files = groups[t]
    files_sorted = sorted(files, key=lambda p: extract_sort_key(p))
    print(f"=== {t} ({len(files_sorted)}) ===")
    for p in files_sorted:
        kind, d1, d2 = extract_sort_key(p)
        size = human_mb(p.stat().st_size)
        rel = p.relative_to(tile_dir)
        if kind == "pair":
            print(f"{size:9.2f} MB  {d1}-{d2}  {rel}")
        elif kind == "single":
            print(f"{size:9.2f} MB  {d1}       {rel}")
        else:
            print(f"{size:9.2f} MB            {rel}")
    print()

Found 194 .img files

=== db8mz (2) ===
   616.12 MB  20230720       lztmre_p089r078_20230720_db8mz.img
   616.12 MB  20230829       lztmre_p089r078_20230829_db8mz.img

=== dc4mz (189) ===
    51.34 MB  20151213       lztmre_p089r078_20151213_dc4mz.img
    51.34 MB  20160114       lztmre_p089r078_20160114_dc4mz.img
    51.34 MB  20160302       lztmre_p089r078_20160302_dc4mz.img
    51.34 MB  20160318       lztmre_p089r078_20160318_dc4mz.img
    51.34 MB  20160403       lztmre_p089r078_20160403_dc4mz.img
    51.34 MB  20160419       lztmre_p089r078_20160419_dc4mz.img
    51.34 MB  20160505       lztmre_p089r078_20160505_dc4mz.img
    51.34 MB  20160606       lztmre_p089r078_20160606_dc4mz.img
    51.34 MB  20160622       lztmre_p089r078_20160622_dc4mz.img
    51.34 MB  20160708       lztmre_p089r078_20160708_dc4mz.img
    51.34 MB  20160724       lztmre_p089r078_20160724_dc4mz.img
    51.34 MB  20160809       lztmre_p089r078_20160809_dc4mz.img
    51.34 MB  20160825       lztmre_p089r07

In [7]:
# from collections import defaultdict


# def bytes_to_gb(n: int) -> float:
#     return n / (1024**3)


# def bytes_to_mb(n: int) -> float:
#     return n / (1024**2)


# ext_counts = defaultdict(int)
# ext_bytes = defaultdict(int)

# all_files = [p for p in tile_dir.rglob("*") if p.is_file()]

# for p in all_files:
#     ext = p.suffix.lower() if p.suffix else "(no_ext)"
#     ext_counts[ext] += 1
#     ext_bytes[ext] += p.stat().st_size

# # Sort by total bytes desc
# rows = sorted(ext_bytes.keys(), key=lambda e: ext_bytes[e], reverse=True)

# print(f"Total files: {len(all_files)}")
# print(f"Total size:  {bytes_to_gb(sum(ext_bytes.values())):.3f} GB\n")

# print(f"{'EXT':<10} {'COUNT':>8} {'SIZE (MB)':>12} {'SIZE (GB)':>12}")
# print("-" * 46)
# for ext in rows:
#     print(
#         f"{ext:<10} {ext_counts[ext]:>8} {bytes_to_mb(ext_bytes[ext]):>12.2f} {bytes_to_gb(ext_bytes[ext]):>12.3f}"
#     )

# # Quick focused summary:
# for focus in [".img", ".shp"]:
#     print(
#         f"\nFocus {focus}: count={ext_counts[focus]}, size={bytes_to_gb(ext_bytes[focus]):.3f} GB"
#     )

In [8]:
# largest = sorted(all_files, key=lambda p: p.stat().st_size, reverse=True)[:25]
# for p in largest:
#     rel = p.relative_to(tile_dir)
#     size_mb = p.stat().st_size / (1024**2)
#     print(f"{size_mb:9.2f} MB  {rel}")

In [9]:
exclude = set()

# Any file whose *stem* ends with dc4mz is part of the dc4 product family.
# e.g. lztmre_..._20250709_dc4mz.img 
#      lztmre_..._20250709_dc4mz.hdr
#      lztmre_..._20250709_dc4mz.img.aux.xml  (suffix is .xml but name contains .img.aux.xml)
# We'll exclude all files that start with the same base "....dc4mz".
for dc4_img in tile_dir.rglob("*dc4mz.img"):
    base = dc4_img.name[:-4]  # remove ".img" -> "....dc4mz"
    for p in dc4_img.parent.iterdir():
        if p.is_file() and p.name.startswith(base):
            exclude.add(p)

print("dc4 products found:", len(list(tile_dir.rglob("*dc4mz.img"))))
print("files excluded (dc4 + ancillary):", len(exclude))


dc4 products found: 189
files excluded (dc4 + ancillary): 567


In [10]:
from collections import defaultdict

include = []
for p in tile_dir.rglob("*"):
    if p.is_file() and p not in exclude:
        include.append(p)

def mb(n): return n/(1024**2)
def gb(n): return n/(1024**3)

total_bytes = sum(p.stat().st_size for p in include)
print("Files to zip:", len(include))
print(f"Total size to zip: {gb(total_bytes):.3f} GB")

# quick extension breakdown
ext_counts = defaultdict(int)
ext_bytes = defaultdict(int)
for p in include:
    ext = p.suffix.lower() if p.suffix else "(no_ext)"
    ext_counts[ext] += 1
    ext_bytes[ext] += p.stat().st_size

print("\nEXT           COUNT    SIZE (MB)")
print("-"*30)
for ext in sorted(ext_bytes, key=lambda e: ext_bytes[e], reverse=True):
    print(f"{ext:<10} {ext_counts[ext]:>8} {mb(ext_bytes[ext]):>12.2f}")


Files to zip: 40
Total size to zip: 1.508 GB

EXT           COUNT    SIZE (MB)
------------------------------
.img              5      1540.29
.shp              6         3.23
.dbf              6         0.36
.shx              6         0.05
.xml              5         0.00
.hdr              5         0.00
.prj              6         0.00
.json             1         0.00


In [17]:
import zipfile

zip_path = tile_dir.with_name(f"{tile}_{timeseries_data}_no-dc4.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in include:
        arcname = p.relative_to(tile_dir.parent)  # keeps "{tile}/..." in zip
        z.write(p, arcname=str(arcname))

print("Wrote:", zip_path)
print("Zip size (GB):", zip_path.stat().st_size / (1024**3))


Wrote: /home/jovyan/work-easi-eds/data/compat/files/fc/p089r078_fc_no-dc4.zip
Zip size (GB): 0.1623652307316661


In [12]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()
bad = [n for n in names if "dc4mz" in n.lower()]
print("dc4mz entries in zip:", len(bad))
if bad:
    print("Example bad entries:")
    for n in bad[:20]:
        print(" ", n)


dc4mz entries in zip: 0
